<div style="text-align: center;">
 <img width=700px heigth=20px src="../IMG/exerxise_gpt_med.png" alt="Example image">
 </div>

In [1]:
cfg_small=GPT_2_config_small={
  "vocab":50257,
  "context_len":1024,
  "emb_dim":768,
  "n_head":12,
  "n_layer":12,
  "dropout":0.1,
  "qkv_bias":False
}

In [2]:
cfg_medium=GPT_2_config_medium={
  "vocab":50257,
  "context_len":1024,
  "emb_dim":1024,
  "n_head":16,
  "n_layer":24,
  "dropout":0.1,
  "qkv_bias":False
}

In [3]:
cfg_large=GPT_2_config_large={
  "vocab":50257,
  "context_len":1024,
  "emb_dim":1280,
  "n_head":20,
  "n_layer":36,
  "dropout":0.1,
  "qkv_bias":False
}

In [1]:
cfg_xl=GPT_2_config_xl={
  "vocab":50257,
  "context_len":1024,
  "emb_dim":1600,
  "n_head":25,
  "n_layer":48,
  "dropout":0.1,
  "qkv_bias":False
}

In [3]:
import torch
import torch.nn as nn

In [4]:
class LayerNorm(nn.Module):
  def __init__(self, emb_dim):
    super().__init__()
    self.epsilon=1e-5
    self.shift=nn.Parameter(torch.zeros(emb_dim))
    self.scale=nn.Parameter(torch.ones(emb_dim))
  def forward(self,inp):
    mean = inp.mean(dim=-1, keepdim=True)
    var = inp.var(dim=-1, keepdim=True)
    input_Norm=(inp-mean)/torch.sqrt(var+self.epsilon)
    return input_Norm*self.scale+self.shift

In [5]:
class MultiHeadAttention(nn.Module):
  def __init__(self,d_in,d_out,context_length,num_heads,dropout,qkv_bias=False):
    super().__init__()

    self.d_in=d_in
    self.d_out=d_out
    self.num_head=num_heads
    if(num_heads==0):
      num_heads+=0.000000001
    self.head_dim=d_out//num_heads

    self.w_query=nn.Linear(d_in, d_out, bias=qkv_bias) #layer not the actual wt
    self.w_key=nn.Linear(d_in,d_out,bias=qkv_bias)
    self.w_value=nn.Linear(d_in, d_out,bias=qkv_bias)
    self.out_proj=nn.Linear(d_out,d_out)
    self.context_length=context_length
    self.dropout=nn.Dropout(dropout)

    self.qkv_bias=qkv_bias
    self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))#register_buffer makes PyTorch officially aware of this tensor:here var is mask

  def forward(self,x):
    batch_size,num_token,d_in=x.shape

    keys=self.w_key(x)
    querys=self.w_query(x)
    values=self.w_value(x)

    #split using view    -- converting (batch_size,num_token,d_in) =-> (batch_size,num_token,num_head,head_dim)
    keys=keys.view(batch_size,num_token,self.num_head,self.head_dim)
    querys=querys.view(batch_size,num_token,self.num_head,self.head_dim)
    values=values.view(batch_size,num_token,self.num_head,self.head_dim)

    #taking the transpose and grouping according to the head  converting---> (batch_size,num_token,num_head,head_dim)=-> (batch_size,num_head,num_token,head_dim)
    keys=keys.transpose(1,2)
    querys=querys.transpose(1,2)
    values=values.transpose(1,2)

    atten_scores=querys@keys.transpose(2,3)

    mask_bool=self.mask.bool()[:num_token,:num_token]
    atten_scores.masked_fill(mask_bool,-torch.inf)

    attn_weigth=torch.softmax(atten_scores/keys.shape[-1]**0.5,dim=-1)
    attn_weigth=self.dropout(attn_weigth)

    context_vec=(attn_weigth@values).transpose(1,2)

    context_vec=context_vec.contiguous().view(batch_size,num_token,self.d_out)
    context_vec = self.out_proj(context_vec)

    return context_vec

In [6]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5*x*(1+torch.tanh(torch.sqrt(torch.tensor(2.0/torch.pi))*(x+0.044715*torch.pow(x,3))))

In [7]:
class FeedForward(nn.Module):
  def __init__(self,GPT_2_config):
    super().__init__()
    self.layer=nn.Sequential(
      nn.Linear(GPT_2_config["emb_dim"],4 * GPT_2_config["emb_dim"]), #expansion
      GELU()  ,               #activation
      nn.Linear(4 * GPT_2_config["emb_dim"], GPT_2_config["emb_dim"]) #contraction
    )
  def forward(self,x):
    return self.layer(x)

In [8]:
class Transfomers(nn.Module):
  def __init__(self,cfg):
    super().__init__()
    self.layernorm=LayerNorm(cfg["emb_dim"])
    self.mutihead_atten=MultiHeadAttention(
      d_in=cfg["emb_dim"],
      d_out=cfg["emb_dim"],
      context_length=cfg["context_len"],
      num_heads=cfg["n_head"],
      dropout=cfg["dropout"],
      qkv_bias=False)
    self.dropout=nn.Dropout(cfg["dropout"])
    self.feed_forward=FeedForward(cfg)

  def forward(self,x):
    shortcut=x

    x=self.layernorm.forward(x)
    x=self.mutihead_atten.forward(x)
    x=self.dropout(x)
    x=x+shortcut

    shortcut=x

    x=self.layernorm(x)
    x=self.feed_forward(x)
    x=self.dropout(x)
    x=x+shortcut

    return x

In [9]:
class GPT(nn.Module):
  def __init__(self, cfg):
    super().__init__()

    self.token_emb=nn.Embedding(cfg["vocab"],cfg["emb_dim"])
    self.pos_emb=nn.Embedding(cfg["context_len"],cfg["emb_dim"])
    self.dropout=nn.Dropout(cfg["dropout"])
    self.tranformer_block= nn.Sequential(
            *[Transfomers(cfg) for _ in range(cfg["n_layer"])]
        )
    self.final_norm=LayerNorm(cfg["emb_dim"])
    self.out_head=nn.Linear(cfg["emb_dim"],cfg["vocab"])


  def forward(self,ip_batch):
      batch,seq_len=ip_batch.shape

      token_embd=self.token_emb(ip_batch)
      pos_embd=self.pos_emb(torch.arange(seq_len))

      x=token_embd+pos_embd

      x=self.dropout(x)

      x=self.tranformer_block(x)

      x=self.final_norm(x)

      logit=self.out_head(x)

      return logit


In [12]:
torch.manual_seed(123)

model_small = GPT(cfg_small)
model_medium=GPT(cfg_medium)
model_large=GPT(cfg_large)

In [35]:
total_params_small = sum(p.numel() for p in model_small.parameters())
total_params_medium = sum(p.numel() for p in model_medium.parameters())
total_params_large= sum(p.numel() for p in model_large.parameters())


print(f"total parameter in GPT-2_small is:{total_params_small}")
print(f"total parameter in GPT-2_medium is:{total_params_medium}")
print(f"total parameter in GPT-2_large is:{total_params_large}")

total parameter in GPT-2_small is:163041361
total parameter in GPT-2_medium is:406213713
total parameter in GPT-2_large is:838178897


In [36]:
print("Input Embedding Layer of GPT-2_Small Shape: ", model_small.token_emb.weight.shape)
print("Output Layer of GPT-2_Small Shape: ", model_small.out_head.weight.shape)
print("-----------------------------------------------")

print("Input Embedding Layer of GPT-2-medium Shape: ", model_medium.token_emb.weight.shape)
print("Output Layer of GPT-2_medium Shape: ", model_medium.out_head.weight.shape)
print("-----------------------------------------------")

print("Input Embedding Layer of GPT-2-large Shape: ", model_large.token_emb.weight.shape)
print("Output Layer of GPT-2-large Shape: ", model_large.out_head.weight.shape)

Input Embedding Layer of GPT-2_Small Shape:  torch.Size([50257, 768])
Output Layer of GPT-2_Small Shape:  torch.Size([50257, 768])
-----------------------------------------------
Input Embedding Layer of GPT-2-medium Shape:  torch.Size([50257, 1024])
Output Layer of GPT-2_medium Shape:  torch.Size([50257, 1024])
-----------------------------------------------
Input Embedding Layer of GPT-2-large Shape:  torch.Size([50257, 1280])
Output Layer of GPT-2-large Shape:  torch.Size([50257, 1280])


## Memory of model

In [37]:
#convert to bit assume 32 bit taken by one param so
total_size_small=total_params_small*4 #in bit
total_size_small_in_mb=total_size_small/(1024*1024) #convert to mb (bit->kb->mb)
print(f"total size of GPT-2_small model weight is :{total_size_small_in_mb:.2f}Mb\n")

total_medium_size=total_params_medium*4 #in bit
total_medium_size_in_mb=total_medium_size/(1024*1024*1024) #convert to mb (bit->kb->mb->Gb)
print(f"total size of GPT-2 model weight is :{total_medium_size_in_mb:.2f}Gb\n")


total_large_size=total_params_large*4 #in bit
total_large_size_in_mb=total_large_size/(1024*1024*1024) #convert to mb (bit->kb->mb->Gb)
print(f"total size of GPT-2 model weight is :{total_large_size_in_mb:.2f}Gb")

total size of GPT-2_small model weight is :621.95Mb

total size of GPT-2 model weight is :1.51Gb

total size of GPT-2 model weight is :3.12Gb


## For Extral large gpt-2

In [10]:
model_xl=GPT(cfg_xl)

In [11]:
print("Input Embedding Layer of GPT-2-x-large Shape: ", model_xl.token_emb.weight.shape)
print("Output Layer of GPT-2-x-large Shape: ", model_xl.out_head.weight.shape)

Input Embedding Layer of GPT-2-x-large Shape:  torch.Size([50257, 1600])
Output Layer of GPT-2-x-large Shape:  torch.Size([50257, 1600])


In [12]:
total_params_xl = sum(p.numel() for p in model_xl.parameters())
print(f"total parameter in GPT-2_xl is:{total_params_xl}")

total parameter in GPT-2_xl is:1637688657


In [13]:
total_xlarge_size=total_params_xl*4 #in bit
total_xlarge_size_in_mb=total_xlarge_size/(1024*1024*1024) #convert to mb (bit->kb->mb->Gb)
print(f"total size of GPT-2 model weight is :{total_xlarge_size_in_mb:.2f}Gb")

total size of GPT-2 model weight is :6.10Gb


## Table comparison

In [15]:
import pandas as pd

# Create structured data
data = [
    ["GPT-2 Small", "621.95 MB", 163041361, "[50257, 768]", "[50257, 768]"],
    ["GPT-2 Medium", "1.51 GB", 406213713, "[50257, 1024]", "[50257, 1024]"],
    ["GPT-2 Large", "3.12 GB", 838178897, "[50257, 1280]", "[50257, 1280]"],
    ["GPT-2 XL", "6.10 GB", 1637688657, "[50257, 1600]", "[50257, 1600]"]
]

# Define columns
columns = [
    "Model",
    "Model Size",
    "Total Parameters",
    "Input Embedding Shape",
    "Output Embedding Shape"
]

# Create DataFrame
df = pd.DataFrame(data, columns=columns)

# Display table
df

,Model,Model Size,Total Parameters,Input Embedding Shape,Output Embedding Shape
0,GPT-2 Small,621.95 MB,163041361,"[50257, 768]","[50257, 768]"
1,GPT-2 Medium,1.51 GB,406213713,"[50257, 1024]","[50257, 1024]"
2,GPT-2 Large,3.12 GB,838178897,"[50257, 1280]","[50257, 1280]"
3,GPT-2 XL,6.10 GB,1637688657,"[50257, 1600]","[50257, 1600]"
